# Reduced LCI Extractor tutorial

This notebook uses `relex` from Python to extract the most important elementary flows and their characterization factors from selected ecoinvent activities. A valid ecoinvent licence is required.

## 1. Install and import

Run `python -m pip install -e .` from the repository root first. Put `ECOINVENT_USERNAME` and `ECOINVENT_PWD` in the environment or in a private `.env` file.

In [ ]:
import pandas as pd

from relex.compute_reduced_inventories import get_reduced_inventories
from relex.save import save_reduced_inventory_data
from relex.utils import build_ecoinvent_in_bw, load_input_data

## 2. Set up Brightway

Supported database names are `ecoinvent-3.10.1-cutoff`, `ecoinvent-3.11-cutoff`, and `ecoinvent-3.12-cutoff`. The import is skipped when the database is already in the project.

In [ ]:
bw_project = "relex-tutorial"
database = "ecoinvent-3.12-cutoff"

build_ecoinvent_in_bw(
    bw_project, database, overwrite_lca_databases=False
)

## 3. Load the exercise input

Create an Excel workbook in `data/input/` with the `activities` and `methods` sheets described in the README. Pass its filename without the `.xlsx` extension. The helper returns the activity and impact-category dictionaries expected by `get_reduced_inventories`.

In [ ]:
input_filename = "exercise-01"
input_data = load_input_data(input_filename)

activities = input_data["activities"]
impact_categories = input_data["impact_cat"]

print(f"Loaded {len(activities)} activities and {len(impact_categories)} impact categories")
display(pd.DataFrame(activities).head())
display(pd.DataFrame(impact_categories).head())

## 4. Compute and inspect the reduced data

The cutoff is a proportion between 0 and 1. The default `0.01` keeps the top contributions identified by Brightway.

In [ ]:
reduced_data = get_reduced_inventories(
    bw_project=bw_project,
    activities=activities,
    impact_categories=impact_categories,
    database=database,
    cutoff=0.01,
)

top_emissions = reduced_data["top_emissions_per_activity"]
characterization_factors = reduced_data["flows_cfs"]
display(top_emissions.head())
display(characterization_factors.head())

The result dictionary contains `top_emissions_per_activity` and `flows_cfs`, both pandas DataFrames.

## 5. Save the result

The helper creates `data/output/relex-tutorial-result.xlsx` with `reduced_inventories` and `cfs` sheets.

In [ ]:
save_reduced_inventory_data(
    reduced_data, output_file="relex-tutorial-result"
)